# Edge AI Model Compression & Optimization — Python Lab

**Goal:** build a practical, Python-first pipeline for shrinking and accelerating neural networks for edge deployment.

This notebook covers:

1. **Baseline measurement** — parameters, approximate model size, MACs/FLOPs, latency.
2. **Unstructured pruning** — remove/deactivate individual weights.
3. **Structured pruning** — remove whole channels/filters so hardware can actually benefit.
4. **Knowledge distillation** — train a smaller student to reproduce a larger teacher.
5. **Quantization** — FP32 → INT8/INT4 concepts and a runnable PyTorch PTQ example.
6. **Neural Architecture Search (NAS)** — redistribute capacity instead of merely deleting it.
7. **Compression sequencing** — compare different optimization orders.
8. **Edge compilation/lowering** — ExecuTorch / TVM integration points.
9. **Partitioning** — CPU/GPU/NPU/DSP-style subgraph placement.
10. **Deployment constraints** — RAM, flash, latency, power, accuracy and operator support.
11. **LLM/AGI-scale compression strategies** — what changes for transformers and multimodal models.

> **Important distinction:** pruning changes *which computation exists*; quantization changes *how precisely it is represented*; distillation changes *where model capacity is learned*; NAS changes *the architecture/capacity allocation*; compilation/lowering maps the resulting graph to hardware.


## 0. The complete edge optimization pipeline

A useful mental model is:

```text
              ┌──────────────────────────────┐
              │   Accuracy-capable baseline  │
              │       Teacher / model        │
              └──────────────┬───────────────┘
                             │
                  profile accuracy + cost
                             │
               ┌─────────────▼─────────────┐
               │  Structured architecture  │
               │       / NAS search         │
               └─────────────┬─────────────┘
                             │
                    smaller student
                             │
               ┌─────────────▼─────────────┐
               │ Knowledge distillation    │
               └─────────────┬─────────────┘
                             │
               ┌─────────────▼─────────────┐
               │ Structured pruning        │
               │ channels / heads / blocks │
               └─────────────┬─────────────┘
                             │
               ┌─────────────▼─────────────┐
               │ Quantization              │
               │ FP32 → INT8 / INT4 / FP8  │
               └─────────────┬─────────────┘
                             │
               ┌─────────────▼─────────────┐
               │ Graph optimization        │
               │ fusion / layout / folding │
               └─────────────┬─────────────┘
                             │
               ┌─────────────▼─────────────┐
               │ Partition + lower         │
               │ CPU / GPU / NPU / DSP     │
               └─────────────┬─────────────┘
                             │
               ┌─────────────▼─────────────┐
               │ Edge runtime              │
               │ ExecuTorch / TVM / vendor │
               └───────────────────────────┘
```

There is **no universal ordering**. In production you benchmark candidate sequences because pruning can change quantization sensitivity and compiler fusion opportunities.


In [ ]:
# 1. Environment
# Recommended:
#   pip install torch torchvision torch-pruning torchprofile matplotlib pandas
#
# Optional edge/compiler environments:
#   pip install executorch
#   pip install apache-tvm
#
# Exact edge backend packages vary by target hardware.

import copy
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. A tiny CNN we can actually manipulate

Using a small CNN makes the structural effects visible. The same ideas transfer to ResNet, MobileNet, ViT and transformer blocks.

### What a convolution channel means

For:

```text
Conv2d(in_channels=3, out_channels=32)
```

there are **32 output filters/channels**.

If we remove output channel 17, we have removed:

```text
filter 17
      ↓
feature map 17
      ↓
the corresponding input channel in the next layer
```

That is **structured channel pruning**.

By contrast, setting a single weight such as:

```text
W[17, 2, 3, 3] = 0
```

is **unstructured pruning**. The tensor shape does not shrink, so generic hardware may see little or no speedup.


In [ ]:
class EdgeCNN(nn.Module):
    def __init__(self, c1=32, c2=64, c3=128, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, c1, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(c1)
        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(c2)
        self.conv3 = nn.Conv2d(c2, c3, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(c3)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c3, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).flatten(1)
        return self.fc(x)

model = EdgeCNN().to(device)
model


In [ ]:
# 3. Baseline parameter and approximate storage measurements

def count_parameters(m):
    return sum(p.numel() for p in m.parameters())

def parameter_bytes(m, bits=32):
    return count_parameters(m) * bits // 8

def summarize_model(m, name="model"):
    params = count_parameters(m)
    return {
        "model": name,
        "parameters": params,
        "params_M": params / 1e6,
        "FP32_MB": parameter_bytes(m) / (1024**2),
    }

baseline = summarize_model(model, "baseline")
pd.DataFrame([baseline])


## 3.1 Compression math

For a dense model with `N` parameters:

- FP32 storage ≈ `4N` bytes
- INT8 storage ≈ `N` bytes
- INT4 storage ≈ `0.5N` bytes

Approximate ideal weight-only compression ratios:

```text
FP32 → INT8 : 4×
FP32 → INT4 : 8×
```

Real files are larger because of scales, zero-points, metadata, alignment and sometimes duplicated tensors.

**Pruning is different:** if 30% of weights are zero but the model is still stored as a dense FP32 tensor, storage may remain approximately unchanged. Sparse encoding or physically removing structured dimensions is required to realize the storage/compute benefit.


# 4. Unstructured pruning — individual weights disappear/deactivate

This is the first place to see exactly what pruning means.

`torch.nn.utils.prune` can apply a mask such as:

```text
W = W_original × mask
```

where:

```text
mask[i] = 0 → weight is disabled
mask[i] = 1 → weight remains active
```

The important catch:

> A zero weight is **not automatically a smaller neural network**.

It is sparse computation only if the storage format and kernel/backend exploit that sparsity.


In [ ]:
import torch.nn.utils.prune as prune

pruned = copy.deepcopy(model)

layer = pruned.conv2
before_nonzero = torch.count_nonzero(layer.weight).item()
total = layer.weight.numel()

# Remove 40% of the lowest-magnitude individual weights.
prune.l1_unstructured(layer, name="weight", amount=0.40)

after_nonzero = torch.count_nonzero(layer.weight).item()

print("Total weights:", total)
print("Nonzero weights:", after_nonzero)
print("Observed sparsity:", 1 - after_nonzero / total)

# Make the mask permanent.
prune.remove(layer, "weight")
print("Physical tensor shape:", tuple(layer.weight.shape))


### Visualizing individual weight removal

The left/right comparison below shows the core difference between dense and sparse tensors.

For real edge acceleration, ask:

1. Does the target accelerator support sparse kernels?
2. What sparsity pattern is supported — arbitrary, block, 2:4, etc.?
3. Does the compiler recognize the pattern?
4. Does memory traffic actually fall?
5. Does latency improve on the target device?

Those questions are more important than the percentage of zeros by itself.


In [ ]:
# Visualize one output filter before/after unstructured pruning.
original = model.conv2.weight.detach().cpu()
sparse = pruned.conv2.weight.detach().cpu()

f = 0
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(original[f, 0].numpy(), cmap="gray")
plt.title("Original filter")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(sparse[f, 0].numpy(), cmap="gray")
plt.title("After unstructured pruning")
plt.axis("off")
plt.show()


# 5. Structured channel pruning — the important edge case

For edge deployment, **structured pruning is often more interesting** because the model tensor can actually become smaller.

Suppose:

```text
Conv2d(64 → 128)
```

and we remove 32 output channels:

```text
Conv2d(64 → 96)
```

Then the next convolution changes from:

```text
Conv2d(128 → 256)
```

to:

```text
Conv2d(96 → 256)
```

So the removal propagates through the graph.

This is why channel pruning is a **graph transformation**, not merely setting weights to zero.


In [ ]:
def channel_l1_scores(conv):
    # One score per output channel/filter.
    w = conv.weight.detach()
    return w.abs().sum(dim=(1, 2, 3))

def channels_to_keep(conv, keep_ratio=0.75):
    scores = channel_l1_scores(conv)
    k = max(1, int(len(scores) * keep_ratio))
    return torch.topk(scores, k=k).indices.sort().values

scores = channel_l1_scores(model.conv2)
keep = channels_to_keep(model.conv2, 0.75)

print("Original output channels:", model.conv2.out_channels)
print("Keeping:", len(keep))
print("Pruning:", model.conv2.out_channels - len(keep))
print("Lowest-scoring channels:", torch.argsort(scores)[:10].tolist())


In [ ]:
# A compact, runnable channel-pruning transformation for our sequential CNN.
# This physically constructs a smaller network instead of leaving zero channels behind.

def prune_edgecnn_channels(src, keep1=0.75, keep2=0.75, keep3=0.75):
    # Select channels using L1 filter importance.
    idx1 = channels_to_keep(src.conv1, keep1)
    idx2 = channels_to_keep(src.conv2, keep2)
    idx3 = channels_to_keep(src.conv3, keep3)

    new = EdgeCNN(
        c1=len(idx1), c2=len(idx2), c3=len(idx3),
        num_classes=src.fc.out_features
    ).to(next(src.parameters()).device)

    with torch.no_grad():
        # Conv1: select output filters.
        new.conv1.weight.copy_(src.conv1.weight[idx1])
        new.conv1.bias.copy_(src.conv1.bias[idx1])
        new.bn1.weight.copy_(src.bn1.weight[idx1])
        new.bn1.bias.copy_(src.bn1.bias[idx1])
        new.bn1.running_mean.copy_(src.bn1.running_mean[idx1])
        new.bn1.running_var.copy_(src.bn1.running_var[idx1])

        # Conv2: select output AND the surviving input channels.
        new.conv2.weight.copy_(src.conv2.weight[idx2][:, idx1])
        new.conv2.bias.copy_(src.conv2.bias[idx2])
        new.bn2.weight.copy_(src.bn2.weight[idx2])
        new.bn2.bias.copy_(src.bn2.bias[idx2])
        new.bn2.running_mean.copy_(src.bn2.running_mean[idx2])
        new.bn2.running_var.copy_(src.bn2.running_var[idx2])

        # Conv3.
        new.conv3.weight.copy_(src.conv3.weight[idx3][:, idx2])
        new.conv3.bias.copy_(src.conv3.bias[idx3])
        new.bn3.weight.copy_(src.bn3.weight[idx3])
        new.bn3.bias.copy_(src.bn3.bias[idx3])
        new.bn3.running_mean.copy_(src.bn3.running_mean[idx3])
        new.bn3.running_var.copy_(src.bn3.running_var[idx3])

        # FC input follows conv3 output channels.
        new.fc.weight.copy_(src.fc.weight[:, idx3])
        new.fc.bias.copy_(src.fc.bias)

    return new

structured = prune_edgecnn_channels(model, 0.75, 0.75, 0.75)

comparison = pd.DataFrame([
    summarize_model(model, "dense baseline"),
    summarize_model(structured, "structured channels -25%"),
])
comparison


### Why structured pruning can reduce computation

For a convolution:

```text
MACs ≈ H × W × Cin × Cout × K²
```

So reducing both `Cin` and `Cout` has a multiplicative effect.

If both dimensions are reduced to 75%:

```text
0.75 × 0.75 = 0.5625
```

or roughly **56% of the original convolution MACs** before accounting for changes elsewhere.

This is the key difference from simply zeroing 25% of weights.


# 6. Knowledge distillation — recover accuracy after compression

Distillation does not directly delete weights.

Instead:

```text
large teacher
     │
     ├── logits / soft targets
     ▼
small student
     │
     └── learns task labels + teacher behavior
```

A common objective is:

```text
Loss = α × CE(student, labels)
     + β × KL(student/T, teacher/T) × T²
```

where `T` is the temperature.

This is especially useful after NAS or structured pruning because the smaller student can regain some of the accuracy lost by compression.


In [ ]:
# Distillation implementation.
# This uses synthetic data so the notebook is self-contained.
# Replace this DataLoader with CIFAR-10/ImageNet/custom data in a real project.

teacher = copy.deepcopy(model).eval()
student = copy.deepcopy(structured).train()

def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5):
    hard = F.cross_entropy(student_logits, labels)
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction="batchmean",
    ) * (T * T)
    return alpha * soft + (1 - alpha) * hard

x = torch.randn(16, 3, 32, 32, device=device)
y = torch.randint(0, 10, (16,), device=device)

with torch.no_grad():
    teacher_logits = teacher(x)

student_logits = student(x)
loss = distillation_loss(student_logits, teacher_logits, y)
print("Example distillation loss:", float(loss))


# 7. Quantization — lower numerical precision

Quantization changes representation:

```text
FP32 weight
   ↓
scale + zero-point
   ↓
INT8 representation
```

Two important workflows:

### PTQ — Post Training Quantization
Train FP32 → calibrate → quantize.

Fast and simple, but accuracy can drop.

### QAT — Quantization Aware Training
Train while simulating quantization.

Usually more work, but lets the model adapt to quantization noise.

For edge systems, **hybrid quantization** is often useful: sensitive layers remain higher precision while less-sensitive layers use INT8/INT4.

ExecuTorch currently documents both PTQ/QAT concepts and backend-specific quantization paths; its runtime can lower exported graphs to CPU/GPU/NPU/DSP-style backends. citeturn0search0turn0search2


In [ ]:
# 8. Simple, runnable dynamic quantization example.
# Dynamic quantization is particularly straightforward for Linear layers.
# For Conv-heavy edge models, use a backend-specific static/PT2E/QAT flow.

quantized_student = torch.ao.quantization.quantize_dynamic(
    copy.deepcopy(student).cpu(),
    {nn.Linear},
    dtype=torch.qint8
)

print("FP32 student:", summarize_model(student.cpu(), "student FP32"))
print("Dynamic INT8 Linear:", summarize_model(quantized_student, "student dynamic INT8"))


### Quantization choices

| Technique | Typical idea | Main benefit | Main issue |
|---|---|---|---|
| FP16/BF16 | 16-bit floating point | easy memory/compute reduction | hardware dependent |
| INT8 PTQ | 8-bit weights/activations | mature edge option | calibration sensitivity |
| INT8 QAT | train with quantization effects | better accuracy retention | training cost |
| INT4 weight-only | 4-bit weights | major model-size reduction | kernels/backend and accuracy sensitivity |
| mixed precision | different precision by layer | accuracy/latency tradeoff | more deployment complexity |

For LLMs, weight-only INT4 and mixed-precision approaches become especially important because weights dominate memory. ExecuTorch's current LLM documentation also exposes low-bit TorchAO configurations such as INT4/INT8 and mixed activation/weight modes. citeturn0search1turn0search10


# 9. Neural Architecture Search — redistribute capacity

Pruning asks:

> **Which existing capacity can I remove?**

NAS asks:

> **Where should the remaining capacity go?**

A search space can vary:

```text
channels:   16 / 24 / 32 / 48 / 64
depth:       1 / 2 / 3 / 4
kernel:      3 / 5 / 7
expansion:   2 / 4 / 6
attention:   enabled / disabled
precision:   FP16 / INT8
```

For edge NAS, the objective is often multi-objective:

```text
maximize accuracy
minimize latency
minimize memory
minimize energy
minimize flash size
```

This can be represented as a Pareto problem rather than a single "best" model.


In [ ]:
# 10. Tiny brute-force NAS example.
# This is deliberately simple: a real project may use Optuna, NNI,
# evolutionary search, differentiable NAS, or hardware-in-the-loop search.

from itertools import product

search_space = {
    "c1": [16, 24, 32],
    "c2": [32, 48, 64],
    "c3": [64, 96, 128],
}

candidates = []
for c1, c2, c3 in product(*search_space.values()):
    candidate = EdgeCNN(c1=c1, c2=c2, c3=c3)
    params = count_parameters(candidate)

    # Proxy objective until real hardware measurements exist.
    # Replace this with measured latency/energy on the target edge device.
    estimated_cost = params

    candidates.append({
        "c1": c1, "c2": c2, "c3": c3,
        "params": params,
        "size_MB_FP32": parameter_bytes(candidate) / (1024**2),
        "proxy_cost": estimated_cost
    })

nas_df = pd.DataFrame(candidates).sort_values("proxy_cost")
nas_df.head(10)


### The important NAS extension: hardware-aware search

Do **not** assume:

```text
fewer FLOPs = faster
```

on a real edge device.

Latency depends on:

- accelerator architecture
- memory bandwidth
- cache behavior
- operator/kernel implementation
- tensor layouts
- supported quantization formats
- compiler fusion
- CPU/NPU/GPU partition boundaries
- batch size
- thermal/power state

A good edge NAS loop therefore looks like:

```text
candidate architecture
        ↓
train / weight-sharing estimate
        ↓
export graph
        ↓
compile for target
        ↓
run on hardware
        ↓
latency + RAM + power + accuracy
        ↓
update search
```

That turns NAS into **hardware-aware architecture search** rather than a purely theoretical FLOP search.


# 11. Compression sequencing — compare strategies

A useful experimental matrix is:

```text
A: baseline → quantization
B: baseline → pruning → fine-tune → quantization
C: baseline → distillation → quantization
D: baseline → structured pruning → distillation → quantization
E: NAS → distillation → structured pruning → quantization
F: NAS → quantization-aware training
```

The correct sequence depends on the model and hardware.

A production experiment should record:

```text
accuracy
parameter count
model file size
peak RAM
MACs/FLOPs
cold-start time
P50 latency
P95 latency
throughput
energy/inference
thermal behavior
operator fallback count
```


In [ ]:
# 12. Generic benchmark harness.
# Replace dummy inference with your target dataset and hardware.

@torch.inference_mode()
def benchmark(model, input_shape=(1, 3, 32, 32), runs=100, warmup=10):
    model = model.eval()
    x = torch.randn(*input_shape, device=next(model.parameters()).device)

    for _ in range(warmup):
        _ = model(x)

    if x.is_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter()

    for _ in range(runs):
        _ = model(x)

    if x.is_cuda:
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    return {
        "avg_ms": elapsed / runs * 1000,
        "throughput_per_sec": runs / elapsed
    }

benchmark(model, runs=30)


# 13. Model partitioning — split the graph across edge accelerators

After compression, the model still has to execute.

A conceptual graph may look like:

```text
Input
  │
  ▼
Preprocess ─────────────── CPU
  │
  ▼
Conv/Attention ────────── NPU
  │
  ▼
Unsupported custom op ─── CPU
  │
  ▼
Postprocess ───────────── GPU/CPU
```

This is **partitioning**.

The compiler identifies supported subgraphs and delegates them to hardware-specific backends.

### Python control plane

Python can define/export/configure the graph.

### Lower-level implementation

The actual kernels/runtime often involve C++, C, CUDA, vendor SDKs, or accelerator-specific code.

So Python is absolutely useful for the workflow, but it cannot magically replace a vendor's hardware backend.


## 13.1 ExecuTorch

**ExecuTorch is the PyTorch edge path I would use as the first concrete Python example.**

The documented flow is essentially:

```text
PyTorch model
     ↓
torch.export
     ↓
edge graph
     ↓
quantization / graph transformations
     ↓
partitioner / backend lowering
     ↓
.pte program
     ↓
ExecuTorch runtime
     ↓
device
```

Its documentation explicitly describes export, compilation, quantization, backend delegation, memory planning and runtime execution. citeturn0search2turn0search4

Example skeleton:

```python
# Illustrative — exact APIs can change with the installed ExecuTorch release.

import torch
from torch.export import export

example = (torch.randn(1, 3, 32, 32),)

exported = export(model.eval(), example)

# Then use the target backend's partitioner/lowering configuration.
# Example pattern:
#
# from executorch.exir import to_edge_transform_and_lower
# from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
#
# edge = to_edge_transform_and_lower(
#     exported,
#     partitioner=[XnnpackPartitioner()]
# )
#
# program = edge.to_executorch()
# with open("edge_model.pte", "wb") as f:
#     f.write(program.buffer)

# NOTE:
# Backend APIs are version-specific; pin the ExecuTorch release in production.
```

ExecuTorch's current examples also show XNNPACK lowering and backend-specific PT2E quantization. citeturn0search7


# 14. Apache TVM — the other major Python-first edge/compiler route

TVM is especially interesting if your goal is **compiler-level control and hardware portability**.

Its current tutorials describe:

```text
PyTorch / ONNX / other model
          ↓
      IRModule
          ↓
graph optimizations
          ↓
tensor-program optimization
          ↓
library dispatch / codegen
          ↓
target compilation
          ↓
CPU / GPU / accelerator
```

TVM emphasizes Python-first customization and composable optimization pipelines. citeturn0search11turn0search9

Current TVM tutorials also demonstrate end-to-end optimization of PyTorch ResNet and LLM deployment workflows. citeturn0search5turn0search3


In [ ]:
# TVM integration point — intentionally guarded because TVM is optional.
# This is where Python hands the model to the compiler.

try:
    import tvm
    print("TVM available:", tvm.__version__)
except ImportError:
    print("TVM not installed.")
    print("Install the appropriate Apache TVM build for your platform.")
    print("The actual target-specific compiler/codegen may require additional native dependencies.")

# Conceptual:
#
# exported_program = torch.export.export(model, example_inputs)
# mod = tvm.relax.frontend.torch.from_exported_program(exported_program)
# mod = tvm.relax.get_pipeline("zero")(mod)
# executable = tvm.compile(mod, target="llvm")  # target changes with hardware


# 15. A practical edge optimization controller

This is the architecture I would use for an MLOps/AI-engineering implementation:

```python
class EdgeOptimizer:
    profile()
    search_architecture()
    distill()
    prune_structured()
    quantize()
    export()
    partition()
    compile()
    benchmark_on_device()
    validate_accuracy()
    select_pareto_frontier()
```

The important idea is that **compression is not one algorithm**. It is an optimization loop with measurements at every stage.


In [ ]:
# 16. Python orchestration skeleton

class EdgeOptimizer:
    def __init__(self, model, device="cpu"):
        self.model = model.to(device)
        self.device = device
        self.history = []

    def record(self, stage, model, metrics=None):
        row = summarize_model(model, stage)
        if metrics:
            row.update(metrics)
        self.history.append(row)

    def structured_prune(self, keep=(0.75, 0.75, 0.75)):
        self.model = prune_edgecnn_channels(self.model, *keep)
        self.record("structured_pruned")
        return self.model

    def quantize(self):
        # For a real CNN use a backend-appropriate PT2E/static/QAT workflow.
        self.model = torch.ao.quantization.quantize_dynamic(
            self.model.cpu(), {nn.Linear}, dtype=torch.qint8
        )
        self.record("quantized")
        return self.model

    def report(self):
        return pd.DataFrame(self.history)

optimizer = EdgeOptimizer(copy.deepcopy(model))
optimizer.record("baseline")
optimizer.structured_prune()
optimizer.report()


# 17. LLM / multimodal / "AGI-scale" compression strategy

For very large transformer or multimodal systems, the same principles remain but the objects change.

### Pruning

Instead of only CNN channels:

```text
CNN → filters/channels
Transformer → heads / MLP neurons / blocks / experts
MoE → experts / expert routing capacity
KV cache → tokens / dimensions / precision
```

### Distillation

```text
large teacher
      ↓
smaller dense model
      ↓
task / instruction / reasoning / representation distillation
```

### Quantization

```text
weights       → INT8 / INT4 / FP8 / lower precision
activations   → INT8 / FP8
KV cache      → INT8 / lower precision
embeddings    → lower precision where tolerated
```

### Architecture optimization

```text
dense → MoE
large → small
long context → local/global attention
full KV cache → compressed/sliding-window cache
uniform width → layer-wise width allocation
```

### Edge-specific compilation

```text
attention kernels
        ↓
fused kernels
        ↓
accelerator-specific codegen
        ↓
memory planning
        ↓
operator partitioning
```

For an "AGI" system, think of this as **capacity allocation across the entire inference stack**, not merely shrinking neural-network weights.


# 18. Compression decision matrix

| Technique | Removes capacity? | Changes precision? | Physically shrinks dense tensors? | Can reduce compute? | Accuracy risk | Edge relevance |
|---|---:|---:|---:|---:|---|---|
| Unstructured pruning | Yes | No | Usually no | Backend-dependent | Medium | Medium |
| Structured channel pruning | Yes | No | **Yes** | **Yes** | Medium | **High** |
| Knowledge distillation | Reallocates capacity | No | Through student | Yes | Medium | **High** |
| INT8 PTQ | No | **Yes** | **Yes** | **Yes** on supported HW | Low–Medium | **Very high** |
| INT8 QAT | No | **Yes** | **Yes** | **Yes** | Lower than PTQ in many cases | High |
| INT4 weight-only | No | **Yes** | **Yes** | Backend-dependent | Medium–High | High for LLMs |
| NAS | **Yes / reallocates** | Optional | Yes | Yes | Search-dependent | **Very high** |
| Graph fusion | No | No | Sometimes | **Yes** | Usually low | **Very high** |
| Partitioning | No | No | No | **Yes** | Low | **Very high** |
| Compiler/codegen | No | No | No | **Yes** | Low | **Very high** |

The key lesson:

> **Model compression and model compilation are complementary.**

A 50% smaller model can still be slow if it maps badly to the accelerator. A slightly larger model can sometimes be faster because its operators fuse better or match the hardware's preferred tensor shapes.


# 19. Production experiment checklist

For every candidate model, persist a row like:

```text
model_id
git_sha
dataset_version
teacher_id
student_architecture
pruning_method
pruning_ratio
quantization_scheme
calibration_dataset
compiler_version
runtime_version
target_device
backend
operator_fallbacks
parameter_count
file_size_bytes
peak_ram_bytes
MACs
FLOPs
accuracy
P50_latency_ms
P95_latency_ms
throughput
energy_per_inference
temperature
```

This turns compression into a reproducible **MLOps optimization experiment** rather than a one-off model tweak.

### Recommended next implementation

For your broader Python/MLOps work, the natural progression is:

```text
PyTorch
  ↓
Torch-Pruning / custom structured pruning
  ↓
distillation training
  ↓
TorchAO / PT2E quantization
  ↓
torch.export
  ↓
ExecuTorch or TVM
  ↓
hardware-specific partitioning
  ↓
Prometheus/Grafana benchmark telemetry
  ↓
MLflow experiment + model registry
```

That gives you a genuine **model-compression → compiler → edge-runtime pipeline** rather than just a collection of compression demos.
